In [2]:
import sys, re, json, time, importlib, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import duckdb
import pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.normalize as N

ARTIFACTS = ROOT / "artifacts"
PARQ      = ARTIFACTS / "parquet"
VALID_GT  = ROOT / "notebooks" / "phase2_valid_gt.parquet"

con = duckdb.connect()
con.execute("PRAGMA threads=12")
con.execute("SET memory_limit='10GB'")
con.execute(f"SET temp_directory='{(ARTIFACTS / 'duckdb_tmp').as_posix()}'")

def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"

con.execute(f"CREATE OR REPLACE VIEW tr_s1  AS SELECT * FROM {pq('train_source1')}")
con.execute(f"""CREATE OR REPLACE VIEW tr_tgt AS
    SELECT * FROM {pq('train_source2')} UNION ALL SELECT * FROM {pq('train_source3')}""")

con.execute(f"""
CREATE OR REPLACE TABLE gt_pairs AS
SELECT source1_entity_id AS s1_id, trim(tgt) AS tgt_id
FROM (SELECT source1_entity_id, unnest(string_split(matched_entity_ids, ',')) AS tgt
      FROM {pq('train_ground_truth')}
      WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> '')
""")

con.execute(f"CREATE OR REPLACE TABLE eval_s1 AS SELECT * FROM read_parquet('{(ARTIFACTS / 'eval_s1.parquet').as_posix()}')")
con.execute("""
CREATE OR REPLACE TABLE eval_pairs AS
SELECT s1_id, trim(tgt) AS tgt_id
FROM (SELECT s1_id, unnest(string_split(matched_entity_ids, ',')) AS tgt FROM eval_s1
      WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> '')
""")

def scripts_of(text):
    if not isinstance(text, str):
        return ()
    s = set()
    for ch in text:
        if ord(ch) < 128 or not ch.isalpha():
            continue
        try:
            s.add(unicodedata.name(ch).split()[0])
        except ValueError:
            s.add("UNKNOWN")
    return tuple(sorted(s))

print(con.execute("SELECT (SELECT count(*) FROM gt_pairs), (SELECT count(*) FROM eval_pairs)").fetchone())

(7638365, 69252)


In [3]:
import json, re, time
from collections import Counter, defaultdict
importlib.reload(N)

NONLAT_RE = re.compile(r"[^\x00-\x7F]")
TOK_SPLIT = re.compile(r"[^\s.,;:()\[\]/\\\-|'\"]+")

def raw_tokens(x):
    """Tokens in original script (NFKC, mojibake fixed, zero-width removed, casefolded)."""
    if not isinstance(x, str):
        return []
    x = unicodedata.normalize("NFKC", x)
    x = N.ZERO_WIDTH_RE.sub("", N.fix_mojibake(x)).casefold()
    return TOK_SPLIT.findall(x)

t0 = time.time()
tr = con.execute(f"""
SELECT s.business_name AS s_name, t.business_name AS t_name,
       s.business_address AS s_addr, t.business_address AS t_addr
FROM (
    SELECT p.* FROM gt_pairs p
    ANTI JOIN read_parquet('{VALID_GT.as_posix()}') v ON v.source1_entity_id = p.s1_id
) p
JOIN tr_s1  s ON s.entity_id = p.s1_id
JOIN tr_tgt t ON t.entity_id = p.tgt_id
WHERE s.country = 'India'
  AND (regexp_matches(t.business_name, '[^\\x00-\\x7F]')
       OR regexp_matches(coalesce(t.business_address, ''), '[^\\x00-\\x7F]'))
""").fetchdf()
print(f"train pairs with Indic text: {len(tr):,}  ({time.time() - t0:.0f}s)")

def mine_translit(s_texts, t_texts, s_norm, min_n=3, min_conf=0.6):
    pair_c, aligned_c = Counter(), Counter()
    for s, t in zip(s_texts, t_texts):
        tt = raw_tokens(t)
        st = (s_norm(s) or "").split()
        if not tt or len(tt) != len(st):          # word-for-word alignment only
            continue
        for w, e in zip(tt, st):
            if NONLAT_RE.search(w):
                pair_c[(w, e)] += 1
                aligned_c[w] += 1
    best = {}
    for (w, e), n in pair_c.items():
        if n >= min_n and n / aligned_c[w] >= min_conf and n > best.get(w, ("", 0))[1]:
            best[w] = (e, n)
    return {w: e for w, (e, _) in best.items()}

name_dict = mine_translit(tr.s_name, tr.t_name, N.normalize_name_v2)
addr_dict = mine_translit(tr.s_addr, tr.t_addr, N.normalize_address_v2)
translit = {**addr_dict, **name_dict}             # name mappings take priority
print(f"name entries: {len(name_dict):,} | addr entries: {len(addr_dict):,} | merged: {len(translit):,}")

(ARTIFACTS / "translit_dict.json").write_text(json.dumps(translit, ensure_ascii=False), encoding="utf-8")
for w, e in list(name_dict.items())[:25]:
    print(f"{w:20} -> {e}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train pairs with Indic text: 992,468  (5s)
name entries: 2,891 | addr entries: 18 | merged: 2,908
यूनिवर्सल            -> universal
फूड                  -> food
प्राइवेट             -> private
लिमिटेड              -> limited
रेड                  -> red
इम्पेक्स             -> impex
प्रा                 -> private
लि                   -> limited
बिग                  -> big
इन्वेस्टमेंट्स       -> investments
तिरुपति              -> tirupati
फाइनेंस              -> finance
വിജയ്                -> vijay
കൺസ്ട്രക്ഷൻ          -> construction
പ്രൈവറ്റ്            -> private
ലിമിറ്റഡ്            -> limited
sóciety              -> society
மை                   -> my
டெக்னாலஜி            -> technology
பிரைவேட்             -> private
லிமிடெட்             -> limited
सनराइज               -> sunrise
बेस्ट                -> best
एग्रो                -> agro
साउथ                 -> south


In [4]:
ev = con.execute("""
SELECT s.business_name AS s_name, t.business_name AS t_name
FROM eval_pairs p
JOIN tr_s1  s ON s.entity_id = p.s1_id
JOIN tr_tgt t ON t.entity_id = p.tgt_id
WHERE s.country = 'India' AND regexp_matches(t.business_name, '[^\\x00-\\x7F]')
""").fetchdf()

stats = defaultdict(Counter)
for r in ev.itertuples():
    tt = raw_tokens(r.t_name)
    st = (N.normalize_name_v2(r.s_name) or "").split()
    script = (scripts_of(r.t_name) or ("?",))[0]
    for i, w in enumerate(tt):
        if not NONLAT_RE.search(w):
            continue
        c = stats[script]
        c["tokens"] += 1
        c["covered"] += w in translit
        if len(tt) == len(st):
            c["aligned"] += 1
            c["dict_correct"] += translit.get(w) == st[i]
            c["anyascii_correct"] += N.to_latin(w) == st[i]

rep = pd.DataFrame(stats).T
rep["coverage"]         = rep["covered"] / rep["tokens"]
rep["dict_acc"]         = rep["dict_correct"] / rep["aligned"]
rep["anyascii_acc"]     = rep["anyascii_correct"] / rep["aligned"]
display(rep.sort_values("tokens", ascending=False).round(3))

# Test-side coverage (unlabelled text only: no labels touched)
tst = con.execute(f"""
SELECT business_name FROM {pq('test_source2')} WHERE country = 'India'
  AND regexp_matches(business_name, '[^\\x00-\\x7F]')
UNION ALL
SELECT business_name FROM {pq('test_source3')} WHERE country = 'India'
  AND regexp_matches(business_name, '[^\\x00-\\x7F]')
USING SAMPLE 100000 ROWS (reservoir, 3)
""").fetchdf()
toks = [w for x in tst.business_name for w in raw_tokens(x) if NONLAT_RE.search(w)]
print(f"test Indic token coverage: {sum(w in translit for w in toks) / len(toks):.3f}  ({len(toks):,} tokens)")

,tokens,covered,aligned,dict_correct,anyascii_correct,coverage,dict_acc,anyascii_acc
DEVANAGARI,10363,10168,10363,10168,10168,0.981,0.981,0.981
LATIN,1538,1271,1131,834,884,0.826,0.737,0.782
TELUGU,1437,1415,1437,1415,1420,0.985,0.985,0.988
KANNADA,1329,1310,1329,1310,1310,0.986,0.986,0.986
BENGALI,1277,1252,1277,1252,1252,0.980,0.980,0.980
TAMIL,1221,1211,1221,1210,1210,0.992,0.991,0.991
GUJARATI,1218,1188,1218,1184,1184,0.975,0.972,0.972
MALAYALAM,750,738,750,738,738,0.984,0.984,0.984
ORIYA,335,330,335,330,330,0.985,0.985,0.985
GURMUKHI,242,236,242,236,236,0.975,0.975,0.975


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test Indic token coverage: 0.942  (2,265,580 tokens)


In [5]:
importlib.reload(N)
for s in ["आदित्य वेंचर्स प्राइवेट लिमिटेड", "बिग कंस्ट्रक्शंस लिमिटेड", "ଲୋଟସ୍ ମାର୍କେଟିଂ ଫୁଡ୍ସ ପ୍ରାଇଭେଟ ଲିମିଟେଡ୍",
          "വിഷൻ ഇൻവെസ്റ്റ്മെന്റ്സ്", "ஓம் புராடக்ட்ஸ் பிரைவேட் லிமிடெட்"]:
    print(f"{s[:35]!r:40} -> {N.normalize_name_v2(s)!r}")

'आदित्य वेंचर्स प्राइवेट लिमिटेड'        -> 'aditya ventures private limited'
'बिग कंस्ट्रक्शंस लिमिटेड'               -> 'big constructions limited'
'ଲୋଟସ୍ ମାର୍କେଟିଂ ଫୁଡ୍ସ ପ୍ରାଇଭେଟ ଲିମି'    -> 'lotus marketing phuds praibhet limited'
'വിഷൻ ഇൻവെസ്റ്റ്മെന്റ്സ്'                -> 'vision investments'
'ஓம் புராடக்ட்ஸ் பிரைவேட் லிமிடெட்'      -> 'om products private limited'


In [6]:
from anyascii import anyascii
bad = []
for r in ev.itertuples():
    if (scripts_of(r.t_name) or ("?",))[0] != "LATIN":
        continue
    tt, st = raw_tokens(r.t_name), (N.normalize_name_v2(r.s_name) or "").split()
    if len(tt) != len(st):
        continue
    for w, e in zip(tt, st):
        if NONLAT_RE.search(w) and N.to_latin(w) != e:
            bad.append((r.t_name[:50], w, anyascii(w), N.to_latin(w), e))
print(len(bad))
display(pd.DataFrame(bad[:25], columns=["t_name", "token", "anyascii", "to_latin", "s1_token"]))

247


,t_name,token,anyascii,to_latin,s1_token
0,sámiti seva bharatiya,sámiti,samiti,samiti,bharatiya
1,GORAKHPUR ADVENTURES LÍMlTED,límlted,limlted,limlted,limited
2,Prívate First Trading Limited,prívate,private,private,first
3,Ichiban India Límited Service,límited,limited,limited,private
4,CRUISE PRIVATE LIMITED SÉRVICES,sérvices,services,services,limited
5,LLP Swayambhu Hóspitality,hóspitality,hospitality,hospitality,llp
6,Maxim Bréomasd (India) LLP,bréomasd,breomasd,breomasd,broadband
7,Private Topstar Márbles Limited,márbles,marbles,marbles,private
8,Company Sa International Schóol,schóol,school,school,company
9,Ltd Kmc Drínking,drínking,drinking,drinking,limited


In [7]:
from rapidfuzz import fuzz

ap = tr[tr.t_addr.map(lambda x: isinstance(x, str) and bool(NONLAT_RE.search(x)))]
ap = ap.sample(min(250_000, len(ap)), random_state=0)
print(f"address pairs for mining: {len(ap):,}")

t0 = time.time()
w_c, we_c = Counter(), Counter()
for s, t in zip(ap.s_addr, ap.t_addr):
    tw = {w for w in raw_tokens(t) if NONLAT_RE.search(w) and w not in translit}
    if not tw:
        continue
    se = {e for e in (N.normalize_address_v2(s) or "").split() if len(e) >= 3 and not e.isdigit()}
    for w in tw:
        w_c[w] += 1
        for e in se:
            we_c[(w, e)] += 1
print(f"counted in {time.time() - t0:.0f}s | distinct tokens: {len(w_c):,}")

cands = defaultdict(list)
for (w, e), n in we_c.items():
    if n >= 3 and n / w_c[w] >= 0.5:
        cands[w].append((e, n))

addr_map = {}
for w, lst in cands.items():
    lat = anyascii(w).lower()
    e, n = max(lst, key=lambda x: (fuzz.ratio(lat, x[0]), x[1]))
    if fuzz.ratio(lat, e) >= 50:
        addr_map[w] = e
print(f"address entries mined: {len(addr_map):,}")
for w, e in list(addr_map.items())[:25]:
    print(f"{w:20} -> {e}")

translit_v2 = {**addr_map, **translit}          # name dictionary keeps priority
(ARTIFACTS / "translit_dict.json").write_text(json.dumps(translit_v2, ensure_ascii=False), encoding="utf-8")
print(f"translit_dict.json: {len(translit_v2):,} entries")

address pairs for mining: 250,000
counted in 3s | distinct tokens: 40
address entries mined: 4
xavier’s             -> xavier
oppï¿1⁄2sushmaï¿1⁄2hospital -> 2hospital
waghbillï¿1⁄2gb      -> waghbilli
zonah’s              -> zonah
translit_dict.json: 2,912 entries


In [8]:
eva = con.execute("""
SELECT s.business_address AS s_addr, t.business_address AS t_addr
FROM eval_pairs p
JOIN tr_s1  s ON s.entity_id = p.s1_id
JOIN tr_tgt t ON t.entity_id = p.tgt_id
WHERE s.country = 'India' AND regexp_matches(coalesce(t.business_address, ''), '[^\\x00-\\x7F]')
""").fetchdf()

tot = hit_any = hit_dict = cov = 0
for s, t in zip(eva.s_addr, eva.t_addr):
    se = set((N.normalize_address_v2(s) or "").split())
    for w in raw_tokens(t):
        if not NONLAT_RE.search(w):
            continue
        tot += 1
        cov += w in translit_v2
        hit_any += anyascii(w).lower() in se
        hit_dict += translit_v2.get(w, anyascii(w).lower()) in se
print(f"Indic address tokens: {tot:,}")
print(f"coverage           : {cov / tot:.3f}")
print(f"anyascii-only hit  : {hit_any / tot:.3f}")
print(f"dict + fallback hit: {hit_dict / tot:.3f}")

Indic address tokens: 6,963
coverage           : 1.000
anyascii-only hit  : 0.023
dict + fallback hit: 1.000


In [9]:
importlib.reload(N)
clean = {k: v for k, v in translit_v2.items()
         if len(k) >= 2 and not N._is_latin_word(k) and not N.REPLACEMENT_RE.search(k)}
(ARTIFACTS / "translit_dict.json").write_text(json.dumps(clean, ensure_ascii=False), encoding="utf-8")
print(f"{len(translit_v2):,} -> {len(clean):,} entries")
importlib.reload(N)

2,912 -> 1,332 entries


<module 'src.normalize' from 'D:\\ML Challenge\\6ab10eb3b23ba_student_resource\\student_resource\\src\\normalize.py'>

In [10]:
for s in ["Kolkata Accessories Prívate Ltd", "Pacific Óptimal Freedom LLC", "Établissements Ailes",
          "Rit Médlsacl Centre", "lakesideï¿½habitat road",
          "आदित्य वेंचर्स प्राइवेट लिमिटेड", "ଲୋଟସ୍ ମାର୍କେଟିଂ ଫୁଡ୍ସ ପ୍ରାଇଭେଟ ଲିମିଟେଡ୍", "ஓம் புராடக்ட்ஸ் பிரைவேட் லிமிடெட்"]:
    print(f"{s[:40]!r:44} -> {N.normalize_name_v2(s)!r}")

'Kolkata Accessories Prívate Ltd'            -> 'kolkata accessories private limited'
'Pacific Óptimal Freedom LLC'                -> 'pacific optimal freedom llc'
'Établissements Ailes'                       -> 'etablissements ailes'
'Rit Médlsacl Centre'                        -> 'rit medlsacl centre'
'lakesideï¿½habitat road'                    -> 'lakesidei 1/2habitat road'
'आदित्य वेंचर्स प्राइवेट लिमिटेड'            -> 'aditya ventures private limited'
'ଲୋଟସ୍ ମାର୍କେଟିଂ ଫୁଡ୍ସ ପ୍ରାଇଭେଟ ଲିମିଟେଡ୍'    -> 'lotus marketing phuds praibhet limited'
'ஓம் புராடக்ட்ஸ் பிரைவேட் லிமிடெட்'          -> 'om products private limited'


In [11]:
importlib.reload(N)
FINAL_REFIT = False   # True only for the final test run

excl = "" if FINAL_REFIT else f"""
    ANTI JOIN read_parquet('{VALID_GT.as_posix()}') v ON v.source1_entity_id = p.s1_id"""

t0 = time.time()
mine_df = con.execute(f"""
SELECT s.business_name AS s_name, t.business_name AS t_name,
       s.business_address AS s_addr, t.business_address AS t_addr
FROM (SELECT p.* FROM gt_pairs p {excl}) p
JOIN tr_s1  s ON s.entity_id = p.s1_id
JOIN tr_tgt t ON t.entity_id = p.tgt_id
""").fetchdf()
print(f"mining pairs: {len(mine_df):,}  ({time.time() - t0:.0f}s)")

def base_tokens(x):
    t = N.to_latin(x)                     # includes learned translit + anyascii
    if not t:
        return []
    t = t.replace("g/f", " ground floor ").replace("/", " ").replace("&", " ")
    return [w for w in N.NONALNUM_RE.sub(" ", t).split() if len(w) >= 2]

def mine_canon(s_col, t_col, min_count=25, min_conf=0.3):
    pair_c, var_c = Counter(), Counter()
    for a, b in zip(mine_df[s_col], mine_df[t_col]):
        ta, tb = base_tokens(a), base_tokens(b)
        sa, sb = set(ta), set(tb)
        ua = [t for t in ta if t not in sb]
        ub = [t for t in tb if t not in sa]
        for t in set(ub):
            var_c[t] += 1
        if len(ua) == len(ub) and 0 < len(ua) <= 3:
            for s_tok, t_tok in zip(ua, ub):
                if s_tok[0] == t_tok[0] and not (s_tok.isdigit() or t_tok.isdigit()):
                    pair_c[(t_tok, s_tok)] += 1
    rows = [(v, c, n, n / var_c[v]) for (v, c), n in pair_c.items()
            if n >= min_count and n / var_c[v] >= min_conf]
    df = (pd.DataFrame(rows, columns=["variant", "canonical", "n", "conf"])
            .sort_values(["variant", "n"], ascending=[True, False])
            .drop_duplicates("variant"))
    df = df[~df["canonical"].isin(df["variant"])]
    return df.sort_values("n", ascending=False).reset_index(drop=True)

t0 = time.time()
name_map = mine_canon("s_name", "t_name")
addr_map = mine_canon("s_addr", "t_addr")
print(f"{len(name_map)} name variants | {len(addr_map)} addr variants  ({time.time() - t0:.0f}s)")
display(name_map.head(40)); display(addr_map.head(40))

canon = {"name": dict(zip(name_map.variant, name_map.canonical)),
         "addr": dict(zip(addr_map.variant, addr_map.canonical))}
(ARTIFACTS / "canon_mined.json").write_text(json.dumps(canon, indent=1), encoding="utf-8")
importlib.reload(N)
print("mined file:", N._MINED.exists())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mining pairs: 6,111,288  (16s)
725 name variants | 1129 addr variants  (412s)


,variant,canonical,n,conf
0,private,pvt,57564,0.999323
1,incorporated,inc,32516,0.973854
2,jy,jay,3663,0.509104
3,centre,center,1205,0.962460
4,ce,care,817,0.714786
5,ass0ciates,associates,707,0.715587
6,technology,technologies,588,0.925984
7,hea1th,health,532,0.722826
8,jai,jay,528,0.996226
9,l1c,llc,511,0.960526


,variant,canonical,n,conf
0,st,street,173126,0.493922
1,rd,road,167919,0.498032
2,dr,drive,161853,0.526591
3,ave,avenue,127280,0.505780
4,texas,tx,110858,0.656967
5,ln,lane,70335,0.538627
6,mh,maharashtra,69907,0.383312
7,ohio,oh,63346,0.585724
8,illinois,il,58912,0.615333
9,tennessee,tn,46091,0.607980


mined file: True


In [12]:
importlib.reload(N)

def not_idempotent(canon):
    return {v: (c, canon[c]) for v, c in canon.items() if c in canon and canon[c] != c}

print("name map:", len(N.LEGAL_CANON), "| non-idempotent:", not_idempotent(N.LEGAL_CANON))
print("addr map:", len(N.ADDR_CANON),  "| non-idempotent:", not_idempotent(N.ADDR_CANON))

for s in ["Sri Shiv Solutions Pvt Ltd", "Shiv Solutions Private Limited", "Hea1th C1inic Inc",
          "Health Clinic Incorporated", "C0 Techn0logies", "Delhi Technology Centre", "A1 Motors"]:
    print(f"{s!r:36} -> {N.normalize_name_v2(s)!r}")
for a in ["12 Saint Louis St, Texas", "21B Main Rd, MH", "101A Oak Ave, Indiana"]:
    print(f"{a!r:36} -> {N.normalize_address_v2(a)!r}")

name map: 732 | non-idempotent: {}
addr map: 1139 | non-idempotent: {}
'Sri Shiv Solutions Pvt Ltd'         -> 'sri shiva solutions private limited'
'Shiv Solutions Private Limited'     -> 'shiva solutions private limited'
'Hea1th C1inic Inc'                  -> 'health clinic incorporated'
'Health Clinic Incorporated'         -> 'health clinic incorporated'
'C0 Techn0logies'                    -> 'company technologies'
'Delhi Technology Centre'            -> 'delhi technologies center'
'A1 Motors'                          -> 'a1 motors'
'12 Saint Louis St, Texas'           -> '12 saint louis street tx'
'21B Main Rd, MH'                    -> '21b main road maharashtra'
'101A Oak Ave, Indiana'              -> '101a oak avenue in'
